### Table Name : _**crm_prd_info**_

In [0]:
df = spark.read.table('data_lakehouse.bronze.crm_prd_info')


### Removing extra spaces

In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import *

for fields in df.schema.fields:
    if isinstance(fields.dataType, StringType):
        df =  df.withColumn(fields.name,trim(fields.name))


### Normalization

In [0]:
df = df.withColumn('prd_line', when(upper('prd_line') == 'R','Road')
                   .when(upper('prd_line') == 'S', 'Other Sales')
                   .when(upper('prd_line') == 'M', 'Mountains')
                   .otherwise('N/A')
                   )


In [0]:
df =  df.withColumn('cat_id',regexp_replace(substring('prd_key',1,5),'-','_')).withColumn('prd_key',substring('prd_key',7,length('prd_key')))



In [0]:
df = df.withColumn('prd_cost', coalesce('prd_cost',lit(0)))


### Column Renaming

In [0]:
RENAMED_COLUMNS = {

    'prd_id': 'product_id',
    'cat_id': 'category_id',
    'prd_key': 'product_number',
    'prd_nm': 'product_name',
    'prd_cost': 'product_cost',
    'prd_line': 'product_line',
    'prd_start_dt': 'start_date',
    'prd_end_dt': 'end_date'
    
}

In [0]:
for old_name,new_name in RENAMED_COLUMNS.items():
    df = df.withColumnRenamed(old_name,new_name)


In [0]:
df = df.select(
    "product_id",      
    "category_id",      
    "product_number",      
    "product_name",  
    "product_cost",     
    "product_line",     
    "start_date",       
    "end_date"    )  


In [0]:
df.write.mode('overwrite').format('delta').saveAsTable('data_lakehouse.silver.crm_prd_info')